In [85]:
import torch
from collections import Counter

In [86]:
# split text file into tokens and characters
data = open('taylor_swift.txt', 'r').read().lower()
tokens = list(data)
vocab = list(set(data))

In [87]:
print(len(vocab), 'unique tokens')
vocab[0:10]

69 unique tokens


[';', 'm', '2', '®', ',', '7', 'p', 'í', 'v', '(']

In [88]:
def merge_pairs(tokens, pair_to_merge):
    # merge the pair in the tokens list
    new_tokens = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair_to_merge:
            new_tokens.append(''.join(pair_to_merge))
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1
    return new_tokens

# find all possible pairs of characters in the text, with space never having a character before it, so we never encounter a token like "abc def"
def is_valid_pair(pair):
    return not pair[1].startswith(' ') or pair[1].startswith('.') or pair[1].startswith('\"')

def train(tokens, vocab, num_merges):
    for _ in range(num_merges):
        # count the pairs in the text
        # the only pais we will need are in the tokens list
        pair_counts = Counter(
            pair
            for pair in zip(tokens, tokens[1:])
            if is_valid_pair(pair)
        )
        # print(f"adding {pair_counts.most_common(1)[0]} to tokens")

        vocab += [(''.join(pair_counts.most_common(1)[0][0]))]
        tokens = merge_pairs(tokens, pair_counts.most_common(1)[0][0])
    return vocab

def encode(text, vocab):
    i = 0
    while i < len(text):
        for token in reversed(vocab):
            if text.startswith(token, i):
                yield token
                i += len(token)
                break
        else:
            yield text[i]
            i += 1

In [89]:
vocab = train(tokens, vocab, 5000)
print(len(vocab), 'unique tokens')
print(f"exmple vocab {vocab[-100:-1]}")

5069 unique tokens
exmple vocab [' career".', ' itun', ' history".', ' 2008.\n', ' plans', ' tribute', ' announce', ' "sco', ' "scoop', ' coup', ' movie".', ' conquered', 'eh', ' eyed', ' peas', ' "kanye', ' ms', ' msn', 'bc.', ' thanks', ' beyon', ' kyle', ' vma', ' phot', ' photo', ' dit', 'ian,', ' "bill', ' vena,', ' talks', " 'h", ' nic', ' source".', 'g!', 'cker,', ' things', ' didn', " didn't", ' artist".', ' knopp', ' knopper,', ' charts".', ' "speak', ' sara', ' grammys', ' shelb', ' shelburn', ' shelburne,', ' craig', ' nsai', " nsai's", ' smith,', ' masterpi', ' masterpiec', ' celebr', ' yahoo!.', ' 2011:', ' cbs', ' allen,', 'il".', ' arcade', 'glob', ' j.', ' explains', ' breakup:', ' split', ' leads', ' raises', ' campaign".', ' mans', ' mansfield,', '-o', '-of', ' connect', 'ton,', ' u.k', ' u.k.', ' "grammy', ' payn', ' payne,', ' "country', ' hold', 'fri', ' timel', ' timeline', ' nominees".', ' doyle', ' robbie', 'ard,', 'anford', '-a', ' apartment', ' earth', ' brown

In [92]:
text = "Copy paste of the Wikipedia article on Taylor Swift, as of Feb 16, 2024."
# print the encoded text
encoded_text = list(encode(text.lower(), vocab))
print(encoded_text)

['co', 'p', 'y', ' pa', 'st', 'e', ' of', ' th', 'e', ' wikipedia', ' article', ' on', ' taylor', ' swift,', ' as', ' of', ' fe', 'b', ' 16', ',', ' 2024.']


In [93]:
# turn tokens of encoded text into indices
token_to_index = list(map(lambda token: vocab.index(token), encoded_text)) # this would enter the network through embedding table
print(f"token to index {token_to_index}")
print([vocab[token_to_index[i]] for i in range(len(token_to_index))])

token to index [1757, 6, 43, 802, 90, 66, 142, 225, 66, 2056, 2057, 115, 174, 570, 230, 142, 661, 33, 4294, 4, 1481]
['co', 'p', 'y', ' pa', 'st', 'e', ' of', ' th', 'e', ' wikipedia', ' article', ' on', ' taylor', ' swift,', ' as', ' of', ' fe', 'b', ' 16', ',', ' 2024.']
